# 03 — Going Further

### AI Builders Lab · take-home companion to Project 2

**You already built this model in class.** This notebook does not teach it again — it picks up
where we ran out of time and does the three things we could not fit into the hour:

| | |
|---|---|
| **1** | Look at the digits the model gets **wrong**, and work out why |
| **2** | Feed it a **photo** of real pen-and-paper handwriting, not just a mouse drawing |
| **3** | **Break it on purpose** — change one thing at a time and see what happens |

Then the homework.

---

### First: save your own copy

Click **Copy to Drive** at the top of this page. Otherwise your work is not saved and
everything you do here disappears when you close the tab.

Run cells with **Shift + Enter**, in order, top to bottom.

---
# Setup — rebuild the model  *(one minute, then walk away)*

Colab wipes its storage when a session ends, so the model you saved in class is gone.
The two cells below are exactly what we did together, compressed with no explanation —
load, normalize, build, train.

If you want the reasoning behind any line of it, it is all in
`02_Handwriting_MNIST_ANN.ipynb`. Nothing here is new.

**Run both cells, then go and get a drink.** Training takes about a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

tf.keras.utils.set_random_seed(42)

(X_train, Y_train), (X_test, Y_test) = mnist.load_data()
X_train = X_train / 255.0          # same normalization as in class
X_test  = X_test  / 255.0

print("Training images:", X_train.shape, " Test images:", X_test.shape)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train, Y_train, epochs=5, batch_size=128,
                    validation_data=(X_test, Y_test), verbose=1)

test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=0)
print("\nACCURACY on unseen data:", round(test_accuracy * 100, 2), "%")

You should be back around **97–98 %**, the same as in class.

Now the interesting part.

---
# Part 1 — What does it get WRONG?

Successes teach you very little. Failures teach you a lot.

We have a model that is right about 97 times out of 100. So let's go and find the other three.

In [ ]:
# Ask the model about all 10,000 test images at once
predictions = model.predict(X_test, verbose=0)

guesses = np.argmax(predictions, axis=1)     # the model's answer for every test image
wrong   = np.where(guesses != Y_test)[0]     # positions where it was wrong

print("The model got", len(wrong), "out of 10,000 wrong.\n")

plt.figure(figsize=(14, 3.5))
for n, i in enumerate(wrong[:10]):
    plt.subplot(1, 10, n + 1)
    plt.imshow(X_test[i], cmap='gray')
    plt.title("is " + str(Y_test[i]) + "\nsaid " + str(guesses[i]), fontsize=9)
    plt.axis('off')
plt.suptitle("Ten mistakes")
plt.show()

**Look at these carefully — this is a real discussion, not a formality.**

Some of these you would probably get wrong too. Some are genuinely ambiguous scrawls.
A few are arguably mislabelled in the original dataset.

This matters because it sets a realistic expectation for every AI system you will ever build:
**the ceiling on accuracy is usually the data, not the model.** If humans disagree about
what the answer is, no neural network is going to hit 100%.

### One more question worth asking

Being wrong is one thing. Being *confidently* wrong is another.

Run the cell below to see how sure the model was about each of its mistakes.

In [ ]:
# reuse the predictions we already made above
confidence = predictions[wrong, guesses[wrong]]      # how sure it was about its WRONG answer

print("Of its", len(wrong), "mistakes:")
print("  confidently wrong (over 90% sure): ", int((confidence > 0.90).sum()))
print("  uncertain          (under 60% sure):", int((confidence < 0.60).sum()))

plt.figure(figsize=(7, 3.5))
plt.hist(confidence * 100, bins=20, color='#3B7DC4')
plt.title("How sure was the model when it was wrong?")
plt.xlabel("confidence in the wrong answer (%)"); plt.ylabel("how many")
plt.grid(alpha=0.3); plt.show()

**Sit with that histogram for a second.**

Some of those mistakes were made at over 99 % confidence. The model was certain, and it was
wrong, and nothing in its output would have warned you.

That is the single most important thing to understand about deployed AI systems.
**A confidence score is not a truth score.** It tells you how strongly the pattern matched
what the model was trained on — nothing more. Keep that in mind every time you see a chatbot
state something with total assurance.

---
# Part 2 — Make it read *your* handwriting

Everything so far used somebody else's data. Now we point the model at you.

There is a problem to solve first, and it is the most useful practical lesson in this notebook.

### The model is extremely fussy about its input

It has only ever seen images that are:

- exactly **28 x 28** pixels
- **white ink on a black background** — not black ink on white paper
- **centred**, and sized to fill about 20 of the 28 pixels
- with pixel values between **0 and 1**

Your drawing or photo will be none of those things.

Feed it in raw and the model will confidently return garbage. **It will not raise an error.
It will just be wrong.** That silence is exactly what makes this failure mode dangerous —
and it is the same rule from class: *whatever you did to the training data, do to your own
data too.*

So we write a function that converts any picture of a digit into MNIST's exact format.
Read the six steps in the comments. **Step 5 — centring on the centre of mass — is the one
everybody skips, and it is the number one reason "it works on MNIST but not on my
handwriting".**

In [ ]:
from PIL import Image

def prepare_digit(img, show=True):
    """Convert ANY picture of a single digit into the 28x28 format the model expects."""

    original = img
    g = np.array(img.convert("L"), dtype=np.float32)   # "L" = convert to greyscale

    # STEP 1: Flip black-on-white to white-on-black.
    # MNIST is white ink on black paper. Your notebook paper is the opposite.
    # We check the brightness of the border pixels: if the edge of the picture is
    # light, we are looking at paper, so we invert.
    border = np.concatenate([g[0, :], g[-1, :], g[:, 0], g[:, -1]])
    if border.mean() > 127:
        g = 255.0 - g

    # STEP 2: Stretch the contrast and delete the faint stuff.
    # Photos have shadows, paper texture, and grey smudges. Anything dimmer than
    # 40% of the brightest ink is treated as background and set to pure black.
    g = g - g.min()
    if g.max() > 0:
        g = g / g.max() * 255.0
    g[g < 0.4 * g.max()] = 0

    # STEP 3: Crop away the empty space, keeping only the ink.
    ys, xs = np.nonzero(g)
    if len(ys) == 0:
        print("I can't find any ink in this image. Try drawing darker or thicker.")
        return np.zeros((28, 28), dtype=np.float32)
    g = g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    # STEP 4: Resize so the longest side is 20 pixels.
    # Why 20 and not 28? Because the people who built MNIST scaled every digit into
    # a 20x20 box and left a 4-pixel margin. We copy them exactly. Get this wrong
    # and your digit is the wrong size compared to everything the model studied.
    h, w = g.shape
    scale = 20.0 / max(h, w)
    new_h, new_w = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    g = np.array(Image.fromarray(g.astype(np.uint8)).resize((new_w, new_h), Image.LANCZOS),
                 dtype=np.float32)

    # STEP 5: Paste into a 28x28 black square, centred by CENTRE OF MASS.
    # MNIST centres each digit on its centre of gravity, not its bounding box.
    # This is the step everyone skips, and skipping it is the #1 reason
    # "it works on MNIST but not on my own writing".
    canvas = np.zeros((28, 28), dtype=np.float32)
    top, left = (28 - new_h) // 2, (28 - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = g
    cy, cx = np.array(np.nonzero(canvas)).mean(axis=1)
    canvas = np.roll(canvas, int(round(13.5 - cy)), axis=0)
    canvas = np.roll(canvas, int(round(13.5 - cx)), axis=1)

    # STEP 6: Scale to 0-1, exactly as we did to the training data.
    canvas = canvas / 255.0

    if show:
        plt.figure(figsize=(7, 3))
        plt.subplot(1, 2, 1); plt.imshow(original, cmap='gray')
        plt.title("What you gave it"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(canvas, cmap='gray')
        plt.title("What the model actually sees"); plt.axis('off')
        plt.show()

    return canvas


def predict_digit(img28):
    """Feed a prepared 28x28 image to the model and show the verdict."""
    p = model.predict(img28.reshape(1, 28, 28), verbose=0)[0]
    guess = int(np.argmax(p))

    print("=" * 44)
    print("   THE MODEL SAYS:", guess, "  (", round(p[guess] * 100, 1), "% confident )")
    print("=" * 44)
    print("\nIts full opinion:")
    for digit in range(10):
        bar = "#" * int(p[digit] * 40)
        print(f"  {digit} | {bar:<40} {p[digit]*100:5.1f}%")
    return guess

print("Helper functions ready.")

## 2a — Draw a digit, right here in this page

Run the cell below. A black box appears **inside the notebook**. Draw one digit, click
**DONE**, and the model answers straight away — one cell, one button, nothing uploaded
anywhere.

- Draw **big** — fill most of the box.
- Draw **thick and confidently**. Thin scratchy lines vanish when we shrink to 28x28.
- One digit only. Click **Clear** to start over.
- Mouse, trackpad, finger or stylus all work.

In [ ]:
# ONE cell: draw, click DONE, and the model answers. Nothing leaves this page.

from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io

canvas_html = '''
<div style="font-family: sans-serif; color:#10162F;">
  <canvas id="pad" width="300" height="300"
          style="border:3px solid #555; background:#000; cursor:crosshair; touch-action:none;"></canvas>
  <div style="margin-top:10px;">
    <button id="done"  style="font-size:16px; padding:9px 20px; cursor:pointer;">DONE &mdash; read my digit</button>
    <button id="clear" style="font-size:16px; padding:9px 20px; cursor:pointer;">Clear</button>
  </div>
  <p style="font-size:13px; color:#666; margin-top:8px;">
     Draw <b>one</b> digit, <b>big</b> and <b>thick</b>, filling most of the box.
     Mouse, trackpad, finger or stylus all work.
  </p>
</div>
<script>
  var c   = document.getElementById('pad');
  var ctx = c.getContext('2d');
  ctx.fillStyle = 'black';
  ctx.fillRect(0, 0, c.width, c.height);
  ctx.strokeStyle = 'white';
  ctx.lineWidth   = 22;          // thick, so the stroke survives shrinking to 28x28
  ctx.lineCap     = 'round';
  ctx.lineJoin    = 'round';

  var drawing = false;
  function spot(e) {
    var r = c.getBoundingClientRect();
    return [(e.clientX - r.left) * c.width / r.width,
            (e.clientY - r.top)  * c.height / r.height];
  }
  // pointer events cover mouse, trackpad, touchscreen and stylus in one go
  c.addEventListener('pointerdown', function(e) {
    e.preventDefault();
    drawing = true;
    var p = spot(e);
    ctx.beginPath(); ctx.moveTo(p[0], p[1]); ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  c.addEventListener('pointermove', function(e) {
    if (!drawing) return;
    e.preventDefault();
    var p = spot(e);
    ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  window.addEventListener('pointerup', function() { drawing = false; });

  document.getElementById('clear').onclick = function() {
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, c.width, c.height);
  };

  // Python waits on this promise until you click DONE
  var data = new Promise(function(resolve) {
    document.getElementById('done').onclick = function() {
      resolve(c.toDataURL('image/png'));
    };
  });
</script>
'''

display(HTML(canvas_html))

# Python pauses here until you click DONE. The timeout means it can never hang forever:
# if nobody clicks, it gives up after 10 minutes instead of freezing the notebook.
def wait_for_drawing():
    try:
        return eval_js("data", timeout_sec=600)
    except TypeError:
        return eval_js("data")      # older Colab builds have no timeout option

try:
    drawing_data = wait_for_drawing()
except Exception as error:
    drawing_data = None
    print("Nothing came back from the drawing pad.")
    print("You probably did not click DONE, or the cell was stopped.")
    print("Just run this cell again and draw another digit.")
    print("  (technical detail:", type(error).__name__, ")")

if drawing_data:
    my_drawing = Image.open(io.BytesIO(b64decode(drawing_data.split(',')[1])))

    if np.array(my_drawing.convert("L")).max() < 10:
        print("The pad was empty. Run this cell again, draw a digit, THEN click DONE.")
    else:
        prepared = prepare_digit(my_drawing)   # squeeze it into MNIST's exact format
        predict_digit(prepared)                # and ask the model

**Did it get it right?**

**Run the cell again** and draw a different digit. Try to find one it fails on.

When it does fail, look at the right-hand picture — *what the model actually sees*.
Nine times out of ten the failure is visible right there: the stroke got too thin,
or the digit was tiny in a corner, or you drew a shape genuinely unlike MNIST's style
(a European 7 with a crossbar, a 1 with a big flag and a base serif, an open-topped 4).

That is not a bug. That is the model honestly telling you it has never seen handwriting like yours.
**A model can only recognise what it has been shown.** Remember that sentence — it explains most
AI failures you will read about in the news.

## 2b — Now use a photo of real handwriting

This is the part we could not do in class, and it is a much harder test than the mouse canvas.

1. Write **one large digit** on white paper with a **thick dark pen or marker** — not pencil,
   it is too faint.
2. Photograph it straight on, in good light, and **crop tightly** around the digit.
3. Run the cell below and click **Choose Files**.

The same `prepare_digit` function handles it. That is why we wrote it to be general rather
than hard-coding it to the canvas.

**Do all ten digits while you are here** — you need them for the homework.

In [ ]:
from google.colab import files

uploaded = files.upload()          # opens a file picker

for filename in uploaded.keys():
    print("\n--- " + filename + " ---")
    photo = Image.open(io.BytesIO(uploaded[filename]))
    prepared = prepare_digit(photo)
    predict_digit(prepared)

**Compare the two.** Your mouse drawings probably scored better than your photos.

That is not a coincidence, and the reason is worth understanding: the canvas already gives us
white-on-black, thick strokes, and a square frame — three quarters of MNIST's format for free.
A photo gives us none of it. Every extra step `prepare_digit` has to guess at is another
chance to guess wrong.

**Real-world data is always messier than the data a model was trained on.** That gap is where
most AI systems fail in practice, and almost none of it is the model's fault.

---
# Part 3 — Break it on purpose

You have a working model. The fastest way to understand it is to damage it and watch what
happens.

Copy the model-building and training cells from the Setup section into the empty cell below,
**change one thing**, re-run, and write down what happened to the test accuracy.

**One change at a time.** If you change three things at once you learn nothing about any of them.

| Change | The question it answers |
|---|---|
| `Dense(128)` → `Dense(16)` | How small can the brain get before it starts failing? |
| `Dense(128)` → `Dense(512)` | Does a bigger brain always help? It costs training time — is it worth it? |
| Delete the `Dropout` layer | Watch the gap between training and test accuracy. Does overfitting appear? |
| `epochs=5` → `epochs=1` | How much does it know after a single pass? |
| `epochs=5` → `epochs=40` | Does it keep improving forever? Where does the test loss turn back up? |
| Delete a whole `Dense` layer | How much does depth matter compared to width? |
| Remove `X_train / 255.0` | What actually happens without normalization? (This one is dramatic.) |

Keep a record as you go — you need it for Problem 1.

In [ ]:
# Your experiment space. Paste, change ONE thing, re-run, record the result.

---
---
# Take it further — optional

**All of this is optional.** Nobody is marking it, there is nothing to hand in, and no one
will ask whether you did it. These are simply the four things worth doing if you want today's
ideas to actually stick, and each one ends in a question the code will not answer for you.
That question is the real exercise.

Do one of them, do all four, or do none. Pick whichever sounds most interesting.

Work through them at whatever pace suits you. If you find something interesting — or something
that breaks — **post it in the class WhatsApp group.** Half the value of a finding is someone
else seeing it.

### 1 — The architecture study

Train **five** different versions of the network, changing one thing each time, and keep a
record:

| # | What I changed | Params | Training accuracy | Test accuracy | Time to train |
|---|---|---|---|---|---|
| 1 | nothing (the baseline above) | 109,386 | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |
| 5 | | | | | |

**Then ask yourself: which change helped most, and which cost the most training time for the
least benefit?** In engineering you are always trading accuracy against cost. Decide what trade
you would make, and be able to say why.

### 2 — Test it on *your* handwriting

Write out all ten digits, 0 through 9, in your own hand. Feed each one through the model —
canvas or photo, your choice.

- What is your model's accuracy on **your** 10 digits?
- How does that compare with the ~97–98 % it scored on MNIST?
- **The gap is the interesting part.** Why does a model that reads 9,700 out of 10,000
  strangers' digits correctly stumble on yours? See if you can name two specific causes, and
  check each one against the "what the model actually sees" picture.

### 3 — Failure analysis

Find **three** digits the model gets wrong — from the test set, from your own handwriting,
or both. For each:

1. Look at the image and the model's full confidence breakdown.
2. Was it *confidently* wrong (over 90 %) or *uncertain* wrong (under 60 %)?
3. What do you think caused it?

Then the question that matters: **which is more dangerous in a real system — a model that is
wrong and knows it, or a model that is wrong and certain?** Think of a real situation where
the difference would matter.

### 4 — Stretch: your first convolutional network

Our network flattens the image into 784 unrelated numbers on the very first line, throwing away
every fact about which pixels sit next to which. A **convolutional** network keeps that spatial
information. Try replacing the first layers with:

```python
layers.Input(shape=(28, 28, 1)),
layers.Conv2D(32, (3, 3), activation='relu'),
layers.MaxPooling2D((2, 2)),
layers.Flatten(),
layers.Dense(64, activation='relu'),
layers.Dense(10, activation='softmax'),
```

What is the new test accuracy, and how long did it take to train? Was the extra time worth it?

*(We cover CNNs properly later in the course — this is a preview. If it works first time, you
are ahead of schedule.)*

---
# Part 11 — When things go wrong

Errors are normal. Every practicing engineer reads error messages all day. Here are the ones
this notebook produces most often.

**`NameError: name 'model' is not defined`**
You skipped a cell, or the session restarted. Run **Runtime → Run all** and wait.

**`NameError: name 'X_train' is not defined`**
Same cause. The data cell has to run before anything that uses the data.

**The accuracy is stuck around 10%**
10% is pure guessing — 1 chance in 10. Almost always this means you ran the normalization cell
(`X_train = X_train / 255.0`) **twice**, so your pixels are now in the range 0 to 0.004. Fix it by
**Runtime → Restart session**, then run everything once, in order.

**My drawing gets predicted wrong every single time**
Look at the "what the model actually sees" picture on the right.

- Is it blank or nearly blank? Draw thicker and darker.
- Is the digit tiny or off in a corner? Something went wrong in cropping — redraw larger.
- Does it look like a reasonable digit but still gets the wrong answer? Then it is a genuine
  disagreement, and it belongs in Homework Problem 3. That is a result, not a bug.

**The photo upload gives nonsense**
Usually lighting. Crop tightly around the digit, use a marker rather than a pencil, avoid shadows
falling across the paper, photograph straight on rather than at an angle.

**`Your session crashed after using all available RAM`**
You probably set a huge layer size or batch size. Restart and go back to sensible numbers.

**Everything is broken and I don't know why**
**Runtime → Restart session and run all.** This fixes a genuinely surprising share of problems,
because it clears out any half-finished state from cells you ran out of order.

---

### What this whole thing was about

You built a neural network from nothing, trained it on 60,000 examples, got it to about 98%
on data it had never seen, made it read your own handwriting, and then went looking for the
places where it breaks.

More importantly, you met the ideas that every project in this course rests on:
**train/test splits, normalization, layers and weights, activation functions, overfitting and
dropout, loss versus accuracy, and the hard truth that a model can only recognise what it has
been shown.**

Everything after this — CNNs, LSTMs, voice, language, reinforcement learning — is a variation
on what you just did.